# ETL Pipeline: Knapsack Problem Instance Processing
**Author**  
Paola A. Castillo-Gutierrez   
A00843399   
**MCC Thesis**  
Exploring a Continual Learning Approach for Algorithm Selection under Distribution Shift in the 0/1 Knapsack Problem


## Overview

This notebook implements an **Extract-Transform-Load (ETL)** pipeline for processing synthetic 0/1 Knapsack Problem (KP) instances generated via the evolutionary-based approach described in Plata-González et al. (2019).

The pipeline performs three stages:

1. **Extract**: Read raw `.kp` instance files from organized subdirectories, parse each file's structure (header metadata + item rows), and assign labels (`dominant_heuristic`, `instance_type`) derived from the filename convention.
2. **Transform**: Compute instance-level normalized features from parsed instances and assemble two DataFrames (item-level and feature-level).
3. **Load**: Produce two consolidated CSV outputs (a unified item-level dataset and an instance-level feature dataset).

## Source File Format (`.kp`)

Each `.kp` file encodes a single 0/1 KP instance with the following structure:

| Row | Column 1 | Column 2 |
|-----|----------|----------|
| 1 (header) | `n` (number of items) | `C` (knapsack capacity) |
| 2 to `n+1` | `w_j` (weight of item `j`) | `p_j` (profit of item `j`) |

The files are organized in subdirectories by experimental set (e.g., `Set-25-32`, `Set-200-256`), where the naming convention is `Set-{n_items}-{capacity}`. 

## Instance Features
### Plata-González et al. (2019) features

Seven normalized features characterize each instance, all bounded to $[0, 1]$ and rounded to 3 decimal places:

| Feature | Symbol | Column | Formula |
|---------|--------|--------|---------|
| Mean weight (normalized) | $\bar{w}$ | `w_mean` | $\text{mean}(w) \;/\; \max(w)$ |
| Median weight (normalized) | $\tilde{w}$ | `w_median` | $\text{median}(w) \;/\; \max(w)$ |
| Std. dev. of weights (normalized) | $\sigma_w$ | `w_std` | $\text{std}(w, \text{ddof}=1) \;/\; \max(w)$ |
| Mean profit (normalized) | $\bar{p}$ | `p_mean` | $\text{mean}(p) \;/\; \max(p)$ |
| Median profit (normalized) | $\tilde{p}$ | `p_median` | $\text{median}(p) \;/\; \max(p)$ |
| Std. dev. of profits (normalized) | $\sigma_p$ | `p_std` | $\text{std}(p, \text{ddof}=1) \;/\; \max(p)$ |
| Weight-profit correlation (shifted) | $r$ | `wp_corr` | $\text{corr}(w, p) \;/\; 2 + 0.5$ |

The standard deviation uses the sample estimator (`ddof=1`), consistent with the paper's worked example (Section 2.2, p. 12713).

The correlation transformation maps Pearson's $r \in [-1, 1]$ to $[0, 1]$.

Additionally, the raw Pearson correlation is stored in `wp_corr_raw` ($r \in [-1, 1]$) to preserve the unshifted value, which matches the paper's worked example directly (see Section 5 below for reconciliation).

### Additional features (not in Plata-González et al., 2019)

| Feature | Column | Formula | Range |
|---------|--------|---------|-------|
| Capacity ratio | `capacity_ratio` | $C \;/\; \sum_{j=1}^{n} w_j$ | $[0, \infty)$ |

This ratio measures the relative tightness of the knapsack constraint. A value close to 0 indicates a highly constrained instance; a value $\geq 1$ means the capacity is non-binding (all items can be packed). This is a standard descriptor in the KP literature for characterizing instance difficulty (Pisinger, 2005).

Four raw range descriptors are also included per instance (not normalized): `weight_min`, `weight_max`, `profit_min`, `profit_max`.

### References (APA 7)

1. Plata-González, L. F., Amaya, I., Ortiz-Bayliss, J. C., Conant-Pablos, S. E., Terashima-Marín, H., & Coello Coello, C. A. (2019). Evolutionary-based tailoring of synthetic instances for the Knapsack problem. *Soft Computing*, *23*, 12711–12728.https://doi.org/10.1007/s00500-019-03822-w

2. Pisinger, D. (2005). Where are the hard knapsack problems? *Computers & Operations Research*, *32*(9), 2271–2284.https://doi.org/10.1016/j.cor.2004.03.002

In [ ]:
import os # Fallback for possible operating-system-level operations
import pandas as pd # Creates and manipulates DataFrames, exports DataFrames to CSV
import numpy as np # Performs numerical computations on arrays
from pathlib import Path # Object-oriented handling of file paths
from dataclasses import dataclass # Automatically generates methods such as __init__ and __repr__ for the class

In [ ]:
# Import the Google Colab drive module and mount it at /content/drive
# This allows accessing files stored in Google Drive from the notebook
# from google.colab import drive
# drive.mount('/content/drive')

## Configuration

In [ ]:
# Base directory (Google Drive) holding the KP instances
# BASE_DIR = Path('/content/drive/MyDrive/KP Instances 2026')

# OUTPUT_DIR = Path('outputs')
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Resolve the repository root so paths work whether the notebook is run
# from the repo root or from the notebooks/ folder
CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD

# BASE_DIR points to the read-only raw KP instances (data/raw)
BASE_DIR = REPO_ROOT / 'data' / 'raw'

# OUTPUT_DIR points to the processed ETL outputs (data/processed/etl_<date>)
OUTPUT_DIR = REPO_ROOT / 'data' / 'processed' / 'etl_2026-02-25'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Domain Model

`KPInstance` is a dataclass that encapsulates the raw data and metadata of a single 0/1 KP instance. The class factory `from_file` handles:

- Parsing the two-column text format (header row + item rows).
- Validating that the declared number of items matches the actual row count.
- Extracting the dominant heuristic and instance difficulty type from the filename convention (e.g., `DEF_EASY_100_000.kp` $\rightarrow$ heuristic=`DEF`, type=`easy`).

In [ ]:
@dataclass # Defines KPInstance as a dataclass to store the information of each KP instance
class KPInstance:
    filename: str # String
    set_name: str # String
    n_items: int  # Integer
    capacity: int # Integer
    weights: np.ndarray #  Array with contiguous memory, efficient for numerical computations, single data type
    profits: np.ndarray #  Array with contiguous memory, efficient for numerical computations, single data type
    dominant_heuristic: str # String
    instance_type: str # String

    @staticmethod # Factory method to create a KP instance from a file
    def from_file(filepath: Path, set_name: str) -> 'KPInstance':
        # Read the file and tokenize each line, stripping commas
        lines = filepath.read_text().strip().splitlines()
        tokens = [[t.strip(',') for t in line.split()] for line in lines if line.strip()]

        # First row: header with n_items and capacity
        n_items = int(tokens[0][0])
        capacity = int(tokens[0][1])

        # Remaining rows: (weight, profit) pairs for each item
        items = np.array(tokens[1:], dtype=float)
        weights = items[:, 0]
        profits = items[:, 1]

        # Validate that the number of items matches the header declaration
        if len(weights) != n_items:
            raise ValueError(
                f"{filepath.name}: header declares {n_items} items "
                f"but file contains {len(weights)} item rows"
            )

        # Extract labels (heuristic and type) from the filename convention
        dominant_heuristic, instance_type = KPInstance._parse_label(filepath.stem)

        return KPInstance(
            filename=filepath.name,
            set_name=set_name,
            n_items=n_items,
            capacity=capacity,
            weights=weights,
            profits=profits,
            dominant_heuristic=dominant_heuristic,
            instance_type=instance_type,
        )

    @staticmethod
    def _parse_label(stem: str) -> tuple:
        # Look for the _EASY or _HARD tag in the filename (case-insensitive)
        # This is not necessary for the data provided for the thesis, but it makes the code more robust against possible variations in the filename format
        stem_upper = stem.upper()
        if '_EASY' in stem_upper:
            tag = '_EASY'
        elif '_HARD' in stem_upper:
            tag = '_HARD'
        else:
            return ('unknown', 'unknown')
        
        # Extract heuristic name (before the tag, preserving case) and instance type (easy/hard)
        idx = stem_upper.index(tag)
        heuristic = stem[:idx]
        instance_type = tag.strip('_').lower()
        return (heuristic, instance_type)

## 1.1. Feature Extractor

`KPFeatureExtractor` computes the seven normalized instance features defined in Plata-González et al. (2019), plus `capacity_ratio` (Pisinger, 2005) and four raw range descriptors.

Logical flow:
1. Cast weight/profit vectors to float and extract their maxima.
2. Guard against degenerate instances where $\max(w) = 0$ or $\max(p) = 0$ (returns all NaN).
3. Compute the six normalized location/dispersion features by dividing by the respective maximum.
4. Compute both the raw Pearson correlation (`wp_corr_raw`) and its $[0,1]$-shifted version (`wp_corr`) in a single call.
5. Compute `capacity_ratio` as $C / \sum w_j$.
6. Validate that all features expected to lie in $[0, 1]$ actually do so.

In [ ]:
class KPFeatureExtractor:
    # Class responsible for extracting and computing 13 normalized features from each KP instance
    # These features are used as input for machine learning models
    
    # Number of decimals to round all normalized features to
    # Used to keep consistency and precision across computations
    DECIMALS = 3

    @classmethod
    def compute(cls, instance: KPInstance) -> dict:
        # MAIN METHOD: Computes all 13 features of a KP instance
        # Input: a KP instance with weights and profits arrays
        # Output: dictionary with 13 computed and normalized features
        # Total features: 6 (dispersion) + 2 (correlation) + 4 (ranges) + 1 (capacity ratio) = 13
        
        # Step 1: Convert weights and profits to float for numerical operations
        w = instance.weights.astype(float)
        p = instance.profits.astype(float)
        # Get the maximum values for later normalization
        max_w = w.max()
        max_p = p.max()

        # Step 2: Guard against degenerate instances
        # If all weights or all profits are zero, normalization is impossible
        # In that case, return a full dictionary of NaN values
        if max_w == 0 or max_p == 0:
            return cls._nan_features()

        # Step 3: Compute the Pearson correlation in both forms (raw and shifted)
        # This correlation measures the linear relationship between weights and profits
        # Note: it is computed ONLY ONCE, with no redundancy
        corr_raw, corr_shifted = cls._shifted_correlation(w, p)

        # Step 4: Build the dictionary with the 13 computed features
        # The features are grouped into 4 groups according to their nature and range
        features = {
            # GROUP 1: Location and dispersion features of weights (3 features)
            # Normalized by dividing by the maximum weight, result in range [0, 1]
            # Rounded to DECIMALS decimals for consistency
            'w_mean':   round(w.mean() / max_w, cls.DECIMALS),          # Normalized mean weight
            'w_median': round(float(np.median(w)) / max_w, cls.DECIMALS),  # Normalized median weight
            'w_std':    round(w.std(ddof=1) / max_w, cls.DECIMALS),    # Standard deviation of weights (ddof=1 for sample)

            # GROUP 2: Location and dispersion features of profits (3 features)
            # Normalized by dividing by the maximum profit, result in range [0, 1]
            # Rounded to DECIMALS decimals for consistency
            'p_mean':   round(p.mean() / max_p, cls.DECIMALS),         # Normalized mean profit
            'p_median': round(float(np.median(p)) / max_p, cls.DECIMALS), # Normalized median profit
            'p_std':    round(p.std(ddof=1) / max_p, cls.DECIMALS),   # Standard deviation of profits (ddof=1 for sample)

            # GROUP 3: Correlation features between weights and profits (2 features)
            # Includes two versions of the same correlation:
            # - Raw version in range [-1, 1] (matches the paper's example)
            # - Normalized version in range [0, 1] (for consistency with the other features)
            'wp_corr_raw': corr_raw,        # Raw Pearson correlation in range [-1, 1]
            'wp_corr':  corr_shifted,       # Normalized Pearson correlation in range [0, 1] (r/2 + 0.5)

            # GROUP 4: UNNORMALIZED range descriptors (4 features)
            # The original values on the absolute scale are kept for reference
            # NOT normalized to [0, 1], they keep their original absolute values
            'weight_min': int(w.min()),     # Minimum weight (absolute value, integer)
            'weight_max': int(max_w),       # Maximum weight (absolute value, integer)
            'profit_min': int(p.min()),     # Minimum profit (absolute value, integer)
            'profit_max': int(max_p),       # Maximum profit (absolute value, integer)

            # GROUP 5: Knapsack constraint measure - relative capacity (1 feature)
            # Ratio = capacity / sum of all weights
            # Values close to 0: very tight knapsack (strong constraint)
            # Values >= 1: non-binding knapsack (all items fit)
            # Not normalized to [0, 1]
            'capacity_ratio': round(instance.capacity / w.sum(), cls.DECIMALS),
        }

        # Step 5: Validate that all normalized features lie in range [0, 1]
        # If any feature is out of range, print a warning for diagnosis
        cls._validate_bounds(features, instance.filename)
        
        # Return the full dictionary with the 13 features
        return features

    @classmethod
    def _shifted_correlation(cls, w: np.ndarray, p: np.ndarray) -> tuple:
        # HELPER METHOD: Computes the Pearson correlation in two formats
        # Input: weights and profits arrays
        # Output: tuple (raw_correlation, normalized_correlation)
        
        # Check whether either vector is constant (all values equal)
        # If constant, the standard deviation is 0 and the correlation is undefined
        if w.std() == 0 or p.std() == 0:
            # Return neutral values (midpoint of each range):
            # - 0.0 for raw correlation (midpoint of [-1, 1])
            # - 0.5 for normalized correlation (midpoint of [0, 1])
            return (0.0, 0.5)
        
        # Compute the raw Pearson correlation ONLY ONCE using NumPy
        # np.corrcoef returns a 2x2 matrix:
        # - [0,0] and [1,1] are 1.0 (self-correlation)
        # - [0,1] and [1,0] are the cross-correlation we need
        r = float(np.corrcoef(w, p)[0, 1])
        
        # Return TWO VERSIONS of the same computed r (without recomputing):
        # 1. r in [-1, 1] rounded to DECIMALS decimals (raw correlation)
        # 2. r/2 + 0.5 in [0, 1] rounded to DECIMALS decimals (normalized correlation)
        # The transformation r/2 + 0.5 linearly maps [-1, 1] to [0, 1]:
        # - r = -1 -> -1/2 + 0.5 = 0.0 (perfect anti-correlation -> minimum)
        # - r = 0 -> 0/2 + 0.5 = 0.5 (no correlation -> middle)
        # - r = 1 -> 1/2 + 0.5 = 1.0 (perfect correlation -> maximum)
        return (round(r, cls.DECIMALS), round(r / 2 + 0.5, cls.DECIMALS))

    @staticmethod
    def _nan_features() -> dict:
        # HELPER METHOD: Returns a dictionary with all 13 features as NaN
        # Used when the instance is degenerate (max_w=0 or max_p=0)
        # This keeps the structure consistent even in problematic cases
        
        # List of 9 normalized features (range [0, 1] under normal conditions)
        # Includes all features expected to be in [0, 1]
        norm_keys = [
            'w_mean', 'w_median', 'w_std',      # 3 from weights dispersion
            'p_mean', 'p_median', 'p_std',      # 3 from profits dispersion
            'wp_corr_raw', 'wp_corr',           # 2 from correlation (both versions)
            'capacity_ratio',                    # 1 from capacity constraint
        ]
        
        # List of 4 unnormalized features (absolute values, integers)
        raw_keys = [
            'weight_min', 'weight_max',         # 2 weight range descriptors
            'profit_min', 'profit_max',         # 2 profit range descriptors
        ]
        
        # Build a dictionary with all normalized features set to NaN
        result = {k: np.nan for k in norm_keys}
        
        # Add the unnormalized features, also set to NaN
        result.update({k: np.nan for k in raw_keys})
        
        # Return the full 13-feature dictionary, all with NaN value
        # This ensures structural consistency even for degenerate instances
        return result

    @staticmethod
    def _validate_bounds(features: dict, filename: str):
        # HELPER METHOD: Validates that normalized features lie in range [0, 1]
        # Runs after computing all features
        # Acts as a quality control to detect problems in the computations
        
        # List of 7 features that MUST lie in range [0, 1]
        # Excluded:
        # - wp_corr_raw, which is in [-1, 1] (different range, not validated here)
        # - capacity_ratio, which can be in [0, infinity) (no upper bound)
        normalized_keys = [
            'w_mean', 'w_median', 'w_std',      # 3 from weights dispersion
            'p_mean', 'p_median', 'p_std',      # 3 from profits dispersion
            'wp_corr',                          # 1 from correlation (normalized version)
        ]
        
        # Iterate over each feature that must lie in [0, 1]
        for key in normalized_keys:
            val = features[key]
            # Check whether the value is outside the valid range [0.0, 1.0]
            # Note: np.isnan(val) returns True if val is NaN, so NaN is ignored naturally
            # The condition "not (0.0 <= val <= 1.0)" is False for NaN, so they are ignored
            if not np.isnan(val) and not (0.0 <= val <= 1.0):
                # If out of range, print a warning with details
                # This helps identify and diagnose normalization problems
                # Format: filename -> feature=value out of [0, 1]
                print(f"  Warning: {filename} -> {key}={val} is out of [0, 1]")

## 2. ETL Pipeline

`KPPipeline` orchestrates the three ETL stages:

- **`extract()`**: Iterates over subdirectories, reads each `.kp` file via `KPInstance.from_file`, and collects the parsed instances. Errors are logged without halting execution.
- **`transform()`**: For each parsed instance, (a) expands item-level rows (one row per weight-profit pair) and (b) computes the feature vector via `KPFeatureExtractor.compute`. Produces two DataFrames with enforced column ordering.
- **`load()`**: Writes both DataFrames to CSV in the specified output directory.
- **`run()`**: Convenience method that chains extract $\rightarrow$ transform $\rightarrow$ load.

In [ ]:
class KPPipeline:
    # Set of valid file extensions accepted as KP instances
    # (.kp is the primary format, .txt and .csv are alternatives)
    VALID_EXTENSIONS = {'.kp', '.txt', '.csv'}

    # Canonical column ordering for the feature DataFrame (instance level)
    # Defines the exact order in which columns will appear in the output CSV file
    FEATURE_COLUMN_ORDER = [
        'filename', 'set', 'dominant_heuristic', 'instance_type',
        'n_items', 'capacity',
        'weight_min', 'weight_max', 'profit_min', 'profit_max',
        'w_mean', 'w_median', 'w_std',
        'p_mean', 'p_median', 'p_std',
        'wp_corr_raw', 'wp_corr',
        'capacity_ratio',
    ]

    # Canonical column ordering for the item DataFrame (item level)
    # Defines the exact order in which columns will appear in the item CSV file
    ITEM_COLUMN_ORDER = [
        'filename', 'set', 'dominant_heuristic', 'instance_type',
        'weight', 'profit',
    ]

    def __init__(self, base_dir: Path):
        # Initialize the pipeline with the base directory containing the instances
        # base_dir: root path containing subfolders of experimental sets
        self.base_dir = base_dir
        # List to store all successfully parsed KP instances
        self.instances: list[KPInstance] = []
        # List to store error messages during extraction (without halting execution)
        self.errors: list[str] = []

    def extract(self) -> 'KPPipeline':
        # PIPELINE STAGE 1: Discover and read KP instances from files
        # Find all subfolders in the base directory (each one represents an experimental set)
        subfolders = sorted(
            [d for d in self.base_dir.iterdir() if d.is_dir()]
        )
        print(f"Found {len(subfolders)} subfolders: {[d.name for d in subfolders]}")

        # Iterate over each subfolder (experimental set)
        for subfolder in subfolders:
            # Within each subfolder, keep only files with valid extensions
            # and sort alphabetically for reproducibility
            files = sorted(
                [f for f in subfolder.iterdir()
                    if f.is_file() and f.suffix in self.VALID_EXTENSIONS]
            )
            count = 0
            # Read each file individually
            for filepath in files:
                try:
                    # Parse the file using KPInstance.from_file()
                    # This method validates the file structure and extracts metadata from the name
                    inst = KPInstance.from_file(filepath, subfolder.name)
                    self.instances.append(inst)
                    count += 1
                except Exception as e:
                    # If an error occurs, log it but keep processing the other files
                    # This provides robustness against malformed files
                    self.errors.append(f"{filepath}: {e}")

            print(f"  {subfolder.name}: {count} instances loaded")

        # Print a summary of the extraction
        print(f"\nTotal: {len(self.instances)} instances, {len(self.errors)} errors")
        if self.errors:
            # Show up to the first 10 errors for diagnosis
            for err in self.errors[:10]:
                print(f"  ERROR: {err}")
        # Return self to allow method chaining (builder pattern)
        return self

    def transform(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        # PIPELINE STAGE 2: Transform instances into structured DataFrames
        # Creates two levels of granularity:
        # 1. Item level: one row per (weight, profit) pair in each instance
        # 2. Feature level: one row per instance with computed features
        
        # Temporary lists to accumulate rows before converting to DataFrames
        item_rows = []      # Rows for the item DataFrame
        feature_rows = []   # Rows for the feature DataFrame

        # Process each extracted KP instance
        for inst in self.instances:
            # PART 1: Expand item-level rows
            # For each (weight, profit) pair in this instance, create a row
            for w, p in zip(inst.weights, inst.profits):
                item_rows.append({
                    'filename': inst.filename,
                    'set': inst.set_name,
                    'dominant_heuristic': inst.dominant_heuristic,
                    'instance_type': inst.instance_type,
                    'weight': int(w),
                    'profit': int(p),
                })

            # PART 2: Compute and add instance-level features
            # Use KPFeatureExtractor to compute the normalized features
            features = KPFeatureExtractor.compute(inst)
            
            # Attach the instance metadata to the features dictionary
            # This links each feature with its source instance
            features.update({
                'filename': inst.filename,
                'set': inst.set_name,
                'dominant_heuristic': inst.dominant_heuristic,
                'instance_type': inst.instance_type,
                'n_items': inst.n_items,
                'capacity': inst.capacity,
            })
            feature_rows.append(features)

        # Build the DataFrames from the lists of dictionaries
        # Apply the canonical column ordering specified in the constants
        df_items = pd.DataFrame(item_rows)[self.ITEM_COLUMN_ORDER]
        df_features = pd.DataFrame(feature_rows)[self.FEATURE_COLUMN_ORDER]

        # Return both DataFrames for use in the next stage
        return df_items, df_features

    def load(self, df_items: pd.DataFrame, df_features: pd.DataFrame,
            output_dir: Path) -> None:
        # PIPELINE STAGE 3: Export the DataFrames to CSV files
        # Define the output paths for the two result files
        items_path = output_dir / 'kp_instances_unified.csv'
        features_path = output_dir / 'kp_features.csv'

        # Export the item DataFrame to CSV
        # index=False avoids saving unnecessary row indices
        df_items.to_csv(items_path, index=False)
        
        # Export the feature DataFrame to CSV
        df_features.to_csv(features_path, index=False)

        # Print export confirmation with information on the amount of data
        print(f"Saved {len(df_items)} item rows -> {items_path}")
        print(f"Saved {len(df_features)} instance features -> {features_path}")

    def run(self, output_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
        # ORCHESTRATES THE FULL ETL CHAIN (Extract -> Transform -> Load)
        # This method runs the three pipeline stages sequentially
        
        # Stage 1: Extract instances from files on disk
        self.extract()
        
        # Stage 2: Transform instances into structured DataFrames
        df_items, df_features = self.transform()
        
        # Stage 3: Load (export) the DataFrames to CSV files
        self.load(df_items, df_features, output_dir)
        
        # Return the DataFrames for later inspection or further analysis
        return df_items, df_features

## 3. Execute Pipeline

In [ ]:
# Create a pipeline instance and run the full ETL chain
pipeline = KPPipeline(BASE_DIR)
df_items, df_features = pipeline.run(OUTPUT_DIR)

## 4. Data Inspection

In [ ]:
# Print summary information of the item-level DataFrame
print("Item-level dataset")
# Show the DataFrame dimensions (rows, columns)
print(f"  Shape: {df_items.shape}")
# List the names of all columns
print(f"  Columns: {list(df_items.columns)}")
# Show the unique heuristic values present in the data
print(f"  Heuristics: {df_items['dominant_heuristic'].unique()}")
# Show the unique instance types (easy/hard)
print(f"  Instance types: {df_items['instance_type'].unique()}")
# Show the first 10 rows of the DataFrame
df_items.head(10)

In [ ]:
# Print summary information of the feature-level DataFrame
print("Feature-level dataset")
# Show the DataFrame dimensions (rows, columns)
print(f"  Shape: {df_features.shape}")
# List the names of all columns
print(f"  Columns: {list(df_features.columns)}")
# Show the range of capacity values (minimum and maximum capacity)
print(f"  Capacity range: [{df_features['capacity'].min()}, {df_features['capacity'].max()}]")
# Show the range of n_items values (minimum and maximum number of items)
print(f"  n_items range: [{df_features['n_items'].min()}, {df_features['n_items'].max()}]")
# Show descriptive statistics (mean, median, std. dev., etc.) of all columns
df_features.describe()

In [ ]:
# List of features to check (all normalized and derived features)
feature_columns = [
    'w_mean', 'w_median', 'w_std',
    'p_mean', 'p_median', 'p_std',
    'wp_corr_raw', 'wp_corr',
    'capacity_ratio',
]

# Count the number of NaN (missing/undefined) values in each feature
nan_count = df_features[feature_columns].isna().sum()

# If at least one feature has NaN values, print the counts
if nan_count.any():
    print("NaN counts per feature:")
    print(nan_count[nan_count > 0])  # Only show features that have NaN
else:
    # If there are no NaN values in any feature, confirm that the data is complete
    print("No NaN values in any feature column.")

## 5. Feature Verification Against Paper Example

Section 2.2 (p. 12713) of Plata-González et al. (2019) provides a worked example:

> "Consider a subset of remaining items whose weights are (2, 2, 3, 4) and whose profits are (10, 5, 6, 15), respectively. Thus, each of the aforementioned features (in order) will take the values of (0.69, 0.63, 0.24, 0.60, 0.53, 0.30, 0.69)."

### Analysis of the first six features ($\bar{w}, \tilde{w}, \sigma_w, \bar{p}, \tilde{p}, \sigma_p$)

These six values are reproducible with `ddof=1` (sample standard deviation) when rounded to 2 decimal places. Using `ddof=0` (population standard deviation) yields $\sigma_w = 0.21$ and $\sigma_p = 0.26$, which do not match the paper's 0.24 and 0.30.

### Analysis of the seventh feature ($r = 0.69$)

The paper defines $r$ as: "weight-profit correlation, divided by two and shifted upwards by 0.5." For the given data:

| Metric | Raw value | After transformation ($\cdot / 2 + 0.5$) |
|--------|-----------|-------------------------------------------|
| Pearson $r$ | 0.689 | 0.845 |
| Spearman $\rho$ | 0.632 | 0.816 |

The paper's reported value of 0.69 matches the raw Pearson correlation without applying the transformation.

### Resolution

Both quantities are now stored explicitly:

| Column | Formula | Range | Interpretation |
|--------|---------|-------|----------------|
| `wp_corr_raw` | $r = \text{corr}(w, p)$ | $[-1, 1]$ | Raw Pearson; reproduces the paper's worked example value of 0.69 |
| `wp_corr` | $r / 2 + 0.5$ | $[0, 1]$ | Normalized per the paper's formula; consistent with the $[0,1]$ bound of all other features |

### Additional feature: `capacity_ratio`

$$\text{capacity\_ratio} = \frac{C}{\sum_{j=1}^{n} w_j}$$

This ratio measures the relative tightness of the knapsack constraint. A value close to 0 indicates a highly constrained instance; a value $\geq 1$ means the capacity is non-binding (all items can be packed). This feature is not part of Plata-González et al. (2019) but is a standard descriptor in the KP literature for characterizing instance difficulty (Pisinger, 2005).


In [ ]:
# Reproduce the worked example from Section 2.2 (p. 12713) of the paper
test_instance = KPInstance(
    filename='verification_example',
    set_name='test',
    n_items=4,
    capacity=10,
    weights=np.array([2, 2, 3, 4], dtype=float),
    profits=np.array([10, 5, 6, 15], dtype=float),
    dominant_heuristic='test',
    instance_type='test',
)

# Compute the features for the test instance
result = KPFeatureExtractor.compute(test_instance)

# Expected values from the paper (rounded to 2 decimals)
paper_values = {
    'w_mean':   (0.69, ''),
    'w_median': (0.63, ''),
    'w_std':    (0.24, ''),
    'p_mean':   (0.60, ''),
    'p_median': (0.53, ''),
    'p_std':    (0.30, ''),
    'wp_corr_raw': (0.69, 'raw Pearson r -- matches the paper example directly'),
    'wp_corr':     (0.845, 'r/2+0.5 -- paper formula; example inconsistency confirmed'),
    'capacity_ratio': (None, 'C / sum(w) = 10/11 ~ 0.909; not in the paper'),
}

# Print the table header with features, computed values and expected values
print(f"{'Feature':<16} {'Computed':>10} {'Paper':>10} {'Match':>6}  Notes")
print('-' * 80)
# Iterate over each feature, comparing computed values with expected ones
for key, (expected, note) in paper_values.items():
    computed = result[key]
    if expected is None:
        match_str = 'N/A'
    else:
        # Determine whether the computed value matches the expected one (tolerance < 0.015)
        match_str = 'Y' if abs(computed - expected) < 0.015 else 'N'
    expected_str = f'{expected:.3f}' if expected is not None else '  ---'
    print(f"{key:<16} {computed:>10.3f} {expected_str:>10} {match_str:>6}  {note}")

# Show the unnormalized range descriptors (minimum and maximum of weights and profits)
print(f"\nRaw range descriptors: weight=[{result['weight_min']}, {result['weight_max']}], "
      f"profit=[{result['profit_min']}, {result['profit_max']}]")
# Show the explanation of the correlation reconciliation between the two formats
print(f"\nCorrelation reconciliation:")
print(f"  wp_corr_raw = {result['wp_corr_raw']:.3f}  -> matches the paper's reported value of 0.69 (raw Pearson r)")
print(f"  wp_corr     = {result['wp_corr']:.3f}  -> correct application of the paper's formula (r/2+0.5)")
print(f"  Both are now stored. ")